In [90]:
from pyspark.sql.functions import col
from pyspark.sql.functions import current_timestamp

In [91]:
%run "04Common.ipynb"

In [92]:
print(f"`{catalog}`.`{silver}`.`silver_roads`")

`sandbox-rey-01`.`silver`.`silver_roads`


In [93]:
def readSilverRoads(catalog, silver):
    print("Reading Silver Table")
    dfSilverRoad = (
        spark.readStream
            .table(f"`{catalog}`.`{silver}`.`silver_roads`")
    )
    print(f"Reading Table Success")
    return dfSilverRoad

def readSilverTraffic(catalog, silver):
    print("Reading Silver Table")
    dfSilverTraffic = (
        spark.readStream
            .table(f"`{catalog}`.`{silver}`.`silver_traffic`")
    )
    print(f"Reading Table Success")
    return dfSilverTraffic

In [94]:
silverRoadsCount = spark.read.table(f"`{catalog}`.`{silver}`.`silver_roads`").count()
silverTrafficCount = spark.read.table(f"`{catalog}`.`{silver}`.`silver_traffic`").count()
print(f"Silver Roads row count: {silverRoadsCount}")
print(f"Silver Traffic row count: {silverTrafficCount}")

Silver Roads row count: 76
Silver Traffic row count: 37054


In [95]:
dfSilverTraffic.schema.names

['Record_ID',
 'Count_point_id',
 'Direction_of_travel',
 'Year',
 'Count_date',
 'hour',
 'Region_id',
 'Region_name',
 'Local_authority_name',
 'Road_name',
 'Road_Category_ID',
 'Start_junction_road_name',
 'End_junction_road_name',
 'Latitude',
 'Longitude',
 'Link_length_km',
 'Pedal_cycles',
 'Two_wheeled_motor_vehicles',
 'Cars_and_taxis',
 'Buses_and_coaches',
 'LGV_Type',
 'HGV_Type',
 'EV_Car',
 'EV_Bike',
 'Extract_Time',
 'Electric_Vehicles_Count',
 'Motor_Vehicles_Count',
 'Transformed_Time']

## Vehicle Intensity

In [96]:
def createVehicleIntensity(df):
    print('Creating Vehicle Intensity', end='')
    
    dfVehicleIntensity = df.withColumn("Vehicle_Intensity",
                col('Motor_Vehicles_Count') / col('Link_length_km')
    )

    print("Success!")
    return dfVehicleIntensity

## LoadTime Column

In [97]:
def createLoadTime(df):
    print('Creating Load Time Column : ', end='')
    df_timestamp = df.withColumn("LoadTime", current_timestamp())

    print("Success!")
    return df_timestamp

## Write to Gold Roads and Traffics

In [98]:
def writeGoldTable(df, catalog, table, folder, queryName):
    print('Writing the silver_roads Data : ',end='') 

    writeStreamGold = (df.writeStream
                .format('delta')
                .option('checkpointLocation',checkpoint+ f"/{folder}/Checkpt/")
                .outputMode('append')
                .queryName(queryName)
                .trigger(availableNow=True)
                .toTable(f"`{catalog}`.`gold`.`{table}`"))
    
    writeStreamGold.awaitTermination()
    print(f'Writing `{catalog}`.`gold`.`{table}` Success!')

## Function Calling

In [99]:
goldRoadsLoads = "GoldRoadsLoads"
goldTrafficLoads = "GoldTrafficLoads"

In [100]:
dfSilverRoad = readSilverRoads(catalog, silver)
dfSilverTraffic = readSilverTraffic(catalog, silver)

dfVehicle = createVehicleIntensity(dfSilverTraffic)
dfGoldTraffic = createLoadTime(dfVehicle)

dfGoldRoad = createLoadTime(dfSilverRoad)

Reading Silver Table
Reading Table Success
Reading Silver Table
Reading Table Success
Creating Vehicle IntensitySuccess!
Creating Load Time Column : Success!
Creating Load Time Column : Success!


In [101]:
dfSilverTraffic.schema.names

['Record_ID',
 'Count_point_id',
 'Direction_of_travel',
 'Year',
 'Count_date',
 'hour',
 'Region_id',
 'Region_name',
 'Local_authority_name',
 'Road_name',
 'Road_Category_ID',
 'Start_junction_road_name',
 'End_junction_road_name',
 'Latitude',
 'Longitude',
 'Link_length_km',
 'Pedal_cycles',
 'Two_wheeled_motor_vehicles',
 'Cars_and_taxis',
 'Buses_and_coaches',
 'LGV_Type',
 'HGV_Type',
 'EV_Car',
 'EV_Bike',
 'Extract_Time',
 'Electric_Vehicles_Count',
 'Motor_Vehicles_Count',
 'Transformed_Time']

In [102]:
#Gold Traffic
writeGoldTable(dfGoldTraffic, catalog, "gold_traffic", goldTrafficLoads, "goldTrafficWriteStream")

#Gold Roads
writeGoldTable(dfGoldRoad, catalog, "gold_roads", goldRoadsLoads, "goldRoadsWriteStream")

Writing the silver_roads Data : Writing `sandbox-rey-01`.`gold`.`gold_traffic` Success!
Writing the silver_roads Data : Writing `sandbox-rey-01`.`gold`.`gold_roads` Success!
